# Artifact and Schema Guard Advanced\ 학습된 Pipeline과 입력 schema·버전·점수 의미를 하나의 artifact로 저장하고, dtype·범주·값 범위·null·무한대·열·운영 mode 위반을 예측 전에 차단  처음부터 설계하는 연습이 더 필요합니다. 강의 문제 원문은 제외

In [1]:
# 공통 준비 코드
from __future__ import annotations

import platform
from pathlib import Path

import imblearn
import joblib
import numpy as np
import pandas as pd
import sklearn
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
n = 1200

# [1] 기본 실습과 같은 혼합형 합성 데이터를 만듭니다.
X = pd.DataFrame({
    "prompt_tokens": rng.integers(20, 1800, n).astype(float),
    "retrieval_score": rng.normal(0.58, 0.18, n).clip(0, 1),
    "toxicity_score": rng.beta(1.5, 8.0, n),
    "route": rng.choice(
        ["chat", "rag", "agent"],
        n,
        p=[0.45, 0.40, 0.15],
    ),
})

latent = (
    -5.0
    + 3.1 * (1 - X["retrieval_score"])
    + 5.0 * X["toxicity_score"]
    + 0.0010 * X["prompt_tokens"]
    + 0.8 * (X["route"] == "agent")
    + rng.normal(0, 0.9, n)
)
y = pd.Series(
    (latent > 0).astype(int),
    name="needs_review",
)

# [2] 학습형 imputer를 확인할 결측값을 입력에 넣습니다.
for column in [
    "prompt_tokens",
    "retrieval_score",
    "toxicity_score",
]:
    X.loc[
        rng.choice(n, size=12, replace=False),
        column,
    ] = np.nan

X.loc[
    rng.choice(n, size=8, replace=False),
    "route",
] = np.nan

X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

NUMERIC_COLUMNS = [
    "prompt_tokens",
    "retrieval_score",
    "toxicity_score",
]
CATEGORICAL_COLUMNS = ["route"]
FEATURE_ORDER = NUMERIC_COLUMNS + CATEGORICAL_COLUMNS

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, NUMERIC_COLUMNS),
    ("cat", categorical_pipeline, CATEGORICAL_COLUMNS),
])

pipeline = ImbPipeline([
    ("preprocessor", preprocessor),
    ("sampler", RandomOverSampler(random_state=RANDOM_STATE)),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_STATE,
        ),
    ),
])

cv = StratifiedKFold(
    n_splits=4,
    shuffle=True,
    random_state=RANDOM_STATE,
)

search = GridSearchCV(
    pipeline,
    {"classifier__C": [0.1, 1.0, 10.0]},
    scoring="average_precision",
    cv=cv,
    n_jobs=1,
    refit=True,
)
search.fit(X_dev, y_dev)
best_pipeline = search.best_estimator_

# [3] 저장·reload와 guard 검증에 공통으로 사용할 대표 online 입력입니다.
# 수치형 열의 null은 이번 schema에서 허용되므로 세 번째 요청은 정상 입력입니다.
new_requests = pd.DataFrame({
    "prompt_tokens": [180.0, 1250.0, np.nan],
    "retrieval_score": [0.91, 0.22, 0.48],
    "toxicity_score": [0.03, 0.51, 0.17],
    "route": ["chat", "agent", "rag"],
})[FEATURE_ORDER]

print("best C:", search.best_params_["classifier__C"])
print("best CV AP:", f"{search.best_score_:.3f}")
print("pipeline steps:", [name for name, _ in best_pipeline.steps])

assert best_pipeline.feature_names_in_.tolist() == FEATURE_ORDER

best C: 1.0
best CV AP: 0.440
pipeline steps: ['preprocessor', 'sampler', 'classifier']


In [2]:
NUMERIC_DOMAIN = {
    "prompt_tokens": (0.0, 4096.0),
    "retrieval_score": (0.0, 1.0),
    "toxicity_score": (0.0, 1.0),
}
CATEGORY_DOMAIN = {
    "route": ["chat", "rag", "agent"],
}
NULLABLE_COLUMNS = NUMERIC_COLUMNS.copy()
SOURCE_MODES = ["batch", "online"]
ONLINE_MAX_ROWS = 128


def build_artifact(best_pipeline, search) -> dict:
    """모델과 입력·재현·출력 계약을 하나의 artifact로 묶습니다."""
    return {
        "model": best_pipeline,
        "model_version": "review-router-1.0",
        "schema_version": "1.0",
        "source_dataset": "synthetic_ai_review_router_v1",
        "feature_order": FEATURE_ORDER,
        "numeric_columns": NUMERIC_COLUMNS,
        "categorical_columns": CATEGORICAL_COLUMNS,
        "numeric_domain": NUMERIC_DOMAIN,
        "category_domain": CATEGORY_DOMAIN,
        "nullable_columns": NULLABLE_COLUMNS,
        "source_modes": SOURCE_MODES,
        "online_max_rows": ONLINE_MAX_ROWS,
        "positive_label": 1,
        "score_meaning": (
            "needs_review 우선순위 점수이며 "
            "보정된 실제 위험 확률을 보장하지 않음"
        ),
        "training": {
            "best_params": search.best_params_,
            "best_cv_AP": float(search.best_score_),
            "cv_folds": 4,
            "scoring": "average_precision",
            "random_state": RANDOM_STATE,
        },
        "versions": {
            "python": platform.python_version(),
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "scikit_learn": sklearn.__version__,
            "imbalanced_learn": imblearn.__version__,
            "joblib": joblib.__version__,
        },
    }


artifact = build_artifact(best_pipeline, search)

# 저장 전에 대표 입력의 점수와 예측을 원본 배열로 보관합니다.
before_classes = artifact["model"].named_steps[
    "classifier"
].classes_.tolist()
before_positive_index = before_classes.index(
    artifact["positive_label"]
)
before_score = artifact["model"].predict_proba(
    new_requests
)[:, before_positive_index]
before_prediction = artifact["model"].predict(new_requests)

# 모델과 schema를 하나의 파일에 저장합니다.
artifact_path = Path("review_pipeline_artifact.joblib")
joblib.dump(artifact, artifact_path)

# 저장 전 메모리 객체가 아니라 실제 파일에서 새 객체를 읽습니다.
loaded = joblib.load(artifact_path)
loaded_classes = loaded["model"].named_steps[
    "classifier"
].classes_.tolist()
loaded_positive_index = loaded_classes.index(
    loaded["positive_label"]
)
after_score = loaded["model"].predict_proba(
    new_requests
)[:, loaded_positive_index]
after_prediction = loaded["model"].predict(new_requests)

score_close = np.allclose(before_score, after_score)
prediction_match = np.array_equal(
    before_prediction,
    after_prediction,
)

print("model/schema version:",
      loaded["model_version"], loaded["schema_version"])
print("feature order:", loaded["feature_order"])
print("artifact keys:", sorted(loaded.keys()))
print("before score:", np.round(before_score, 3).tolist())
print("before prediction:", before_prediction.tolist())
print("reload scores close:", score_close)
print("reload predictions identical:", prediction_match)

assert loaded["model"].feature_names_in_.tolist() == loaded["feature_order"]
assert score_close
assert prediction_match

model/schema version: review-router-1.0 1.0
feature order: ['prompt_tokens', 'retrieval_score', 'toxicity_score', 'route']
artifact keys: ['categorical_columns', 'category_domain', 'feature_order', 'model', 'model_version', 'nullable_columns', 'numeric_columns', 'numeric_domain', 'online_max_rows', 'positive_label', 'schema_version', 'score_meaning', 'source_dataset', 'source_modes', 'training', 'versions']
before score: [0.001, 0.999, 0.286]
before prediction: [0, 1, 0]
reload scores close: True
reload predictions identical: True


In [3]:
from pandas.api.types import is_numeric_dtype, is_string_dtype


def validate_input(
    frame: pd.DataFrame,
    artifact: dict,
    source_mode: str,
) -> pd.DataFrame:
    """예측 전에 입력 schema와 운영 계약을 검사합니다."""
    # DataFrame 여부를 먼저 확인해 columns 접근 오류를 계약 오류와 구분합니다.
    if not isinstance(frame, pd.DataFrame):
        raise TypeError("입력은 pandas DataFrame이어야 합니다.")

    # 지원하는 호출 경로와 online 요청 크기를 모델 실행 전에 확인합니다.
    if source_mode not in artifact["source_modes"]:
        raise ValueError(
            f"허용되지 않은 source_mode: {source_mode}"
        )
    if (
        source_mode == "online"
        and len(frame) > artifact["online_max_rows"]
    ):
        raise ValueError(
            "online 모드는 한 번에 최대 "
            f"{artifact['online_max_rows']}행만 허용합니다."
        )

    expected = artifact["feature_order"]
    received = frame.columns.tolist()

    # 누락·추가 열은 열 순서 오류보다 먼저 구분합니다.
    missing = sorted(set(expected) - set(received))
    extra = sorted(set(received) - set(expected))
    if missing or extra:
        raise ValueError(
            f"스키마 불일치: missing={missing}, extra={extra}"
        )

    # 이번 artifact는 정확한 입력 순서까지 계약으로 고정합니다.
    if received != expected:
        raise ValueError(
            f"열 순서 불일치: expected={expected}, "
            f"received={received}"
        )

    checked = frame.copy()

    # nullable 목록에 없는 필수 열의 null만 거부합니다.
    required_columns = [
        column
        for column in expected
        if column not in artifact["nullable_columns"]
    ]
    if checked[required_columns].isna().any().any():
        raise ValueError(
            "필수 열에는 null을 허용하지 않습니다: "
            f"{required_columns}"
        )

    # 숫자 dtype, finite 여부, domain은 원인이 달라 따로 검사합니다.
    for column in artifact["numeric_columns"]:
        if not is_numeric_dtype(checked[column]):
            raise TypeError(
                f"{column}은 숫자 dtype이어야 합니다."
            )

        # 허용된 null은 제외하고 실제 관측값만 검사합니다.
        values = checked[column].dropna().to_numpy(dtype=float)
        if not np.isfinite(values).all():
            raise ValueError(
                f"{column}에 inf 또는 -inf가 있습니다."
            )

        lower, upper = artifact["numeric_domain"][column]
        if ((values < lower) | (values > upper)).any():
            raise ValueError(
                f"{column} 값이 허용 범위 "
                f"[{lower}, {upper}]를 벗어났습니다."
            )

    # 범주 dtype과 실제 관측 범주를 모두 검사합니다.
    for column in artifact["categorical_columns"]:
        series = checked[column]
        if not (
            is_string_dtype(series.dtype)
            or isinstance(series.dtype, pd.CategoricalDtype)
        ):
            raise TypeError(
                f"{column}은 문자열 또는 category dtype이어야 합니다."
            )

        observed = set(series.astype(str).unique())
        allowed = set(artifact["category_domain"][column])
        unknown = sorted(observed - allowed)
        if unknown:
            raise ValueError(
                f"{column}에 허용되지 않은 범주가 있습니다: "
                f"{unknown}"
            )

    return checked


def predict_with_contract(
    frame: pd.DataFrame,
    artifact: dict,
    source_mode: str,
):
    """검증을 통과한 입력에만 모델 점수와 예측을 만듭니다."""
    checked = validate_input(frame, artifact, source_mode)
    model = artifact["model"]

    classes = model.named_steps[
        "classifier"
    ].classes_.tolist()
    positive_index = classes.index(artifact["positive_label"])

    score = model.predict_proba(checked)[:, positive_index]
    prediction = model.predict(checked)
    return score, prediction


def expect_guard(
    name,
    bad_frame,
    artifact,
    expected_exception,
    mode="batch",
):
    """예상한 예외 종류가 발생할 때만 guard 테스트를 통과시킵니다."""
    try:
        predict_with_contract(
            bad_frame,
            artifact,
            source_mode=mode,
        )
    except expected_exception as error:
        return {
            "guard": name,
            "exception": type(error).__name__,
            "message": str(error),
        }
    except Exception as error:
        # 예외가 발생했더라도 종류가 계약과 다르면 실패입니다.
        raise AssertionError(
            f"{name}: 예상 {expected_exception.__name__}, "
            f"실제 {type(error).__name__}"
        ) from error

    # 잘못된 입력이 예외 없이 통과한 것도 guard 실패입니다.
    raise AssertionError(
        f"{name} guard가 동작하지 않았습니다."
    )


# 심화 1에서 실제 파일로 reload한 artifact를 사용합니다.
# 정상 online 입력이 점수와 예측을 만드는지 먼저 확인합니다.
online_score, online_prediction = predict_with_contract(
    new_requests,
    loaded,
    source_mode="online",
)

# 같은 계약을 batch 경로에서도 확인합니다.
# X_test 전체를 평가 데이터로 다시 쓰는 것이 아니라, 입력 계약 통과 여부와
# 출력 행 수 보존만 확인하기 위한 대표 입력 8행입니다.
valid_batch = X_test.head(8)[FEATURE_ORDER].copy()
batch_score, batch_prediction = predict_with_contract(
    valid_batch,
    loaded,
    source_mode="batch",
)

guard_results = []

# 1. 숫자 열을 문자열 dtype으로 변경합니다.
bad_dtype = new_requests.copy()
bad_dtype["prompt_tokens"] = ["180", "1250", "300"]
guard_results.append(expect_guard(
    "dtype", bad_dtype, loaded, TypeError, mode="online"
))

# 2. 허용 목록에 없는 route를 전달합니다.
bad_category = new_requests.copy()
bad_category["route"] = "tool"
guard_results.append(expect_guard(
    "category", bad_category, loaded, ValueError, mode="online"
))

# 3. 0~1 범위를 벗어난 retrieval_score를 전달합니다.
bad_domain = new_requests.copy()
bad_domain["retrieval_score"] = 1.5
guard_results.append(expect_guard(
    "domain", bad_domain, loaded, ValueError, mode="online"
))

# 4. 필수 범주형 열 route를 비웁니다.
bad_null = new_requests.copy()
bad_null["route"] = None
guard_results.append(expect_guard(
    "required_null", bad_null, loaded, ValueError, mode="online"
))

# 5. 유한하지 않은 숫자 값을 전달합니다.
bad_inf = new_requests.copy()
bad_inf["toxicity_score"] = np.inf
guard_results.append(expect_guard(
    "inf", bad_inf, loaded, ValueError, mode="online"
))

# 6. 필수 열 하나를 제거합니다.
bad_missing = new_requests.drop(columns=["route"])
guard_results.append(expect_guard(
    "missing_column", bad_missing, loaded, ValueError, mode="online"
))

# 7. 학습 계약에 없는 추가 열을 전달합니다.
bad_extra = new_requests.copy()
bad_extra["unexpected_feature"] = 0

guard_results.append(expect_guard(
    "extra_column", bad_extra, loaded, ValueError, mode="online"
))

# 8. 같은 열 집합을 다른 순서로 전달합니다.
bad_order = new_requests[[
    "route",
    "toxicity_score",
    "retrieval_score",
    "prompt_tokens",
]]
guard_results.append(expect_guard(
    "column_order", bad_order, loaded, ValueError, mode="online"
))

# 9. 지원하지 않는 호출 mode를 전달합니다.
guard_results.append(expect_guard(
    "source_mode", new_requests, loaded, ValueError, mode="stream"
))

# 10. online 최대 행 수 128을 초과합니다.
bad_online_size = pd.concat(
    [new_requests.iloc[[0]]] * 129,
    ignore_index=True,
)
guard_results.append(expect_guard(
    "online_size",
    bad_online_size,
    loaded,
    ValueError,
    mode="online",
))

print("online score:", np.round(online_score, 3).tolist())
print("online prediction:", online_prediction.tolist())
print("batch output rows:", len(batch_score))
for result in guard_results:
    print(
        f"{result['guard']}: "
        f"{result['exception']} - {result['message']}"
    )

expected_guards = {
    "dtype",
    "category",
    "domain",
    "required_null",
    "inf",
    "missing_column",
    "extra_column",
    "column_order",
    "source_mode",
    "online_size",
}
assert {result["guard"] for result in guard_results} == expected_guards
assert len(online_score) == len(new_requests)
assert len(online_prediction) == len(new_requests)
assert len(batch_score) == len(valid_batch) == 8
assert len(batch_prediction) == len(valid_batch)

online score: [0.001, 0.999, 0.286]
online prediction: [0, 1, 0]
batch output rows: 8
dtype: TypeError - prompt_tokens은 숫자 dtype이어야 합니다.
category: ValueError - route에 허용되지 않은 범주가 있습니다: ['tool']
domain: ValueError - retrieval_score 값이 허용 범위 [0.0, 1.0]를 벗어났습니다.
required_null: ValueError - 필수 열에는 null을 허용하지 않습니다: ['route']
inf: ValueError - toxicity_score에 inf 또는 -inf가 있습니다.
missing_column: ValueError - 스키마 불일치: missing=['route'], extra=[]
extra_column: ValueError - 스키마 불일치: missing=[], extra=['unexpected_feature']
column_order: ValueError - 열 순서 불일치: expected=['prompt_tokens', 'retrieval_score', 'toxicity_score', 'route'], received=['route', 'toxicity_score', 'retrieval_score', 'prompt_tokens']
source_mode: ValueError - 허용되지 않은 source_mode: stream
online_size: ValueError - online 모드는 한 번에 최대 128행만 허용합니다.
